# 4.最終モデルを構築する
目的変数に値を持たないテストデータの発光波長&量子収率を予測する

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
import pandas as pd量子収率

from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

import lightgbm as lgb

from rdkit import Chem
from rdkit.Chem import Draw

import optuna
import joblib
import os

import optuna

## データ読み込み

### 学習データ読み込み

In [2]:
# 入力：読み込みたい記述子のタイプを選択
descriptor_type = "mordred_3d"  # 'rdkit' or 'mordred_2d' or 'mordred_3d' or 'fp'

dataset = pd.read_csv("../data/dataset_train.csv", index_col=0)
target = dataset[["Emission max (nm)", "Quantum yield"]]

if descriptor_type == "rdkit":
    des = pd.read_csv("outputs/descriptors/rdkit/X_train_rdkit_processed.csv", index_col=0)
elif descriptor_type == "mordred_2d":
    des = pd.read_csv(
        "outputs/descriptors/mordred_2d/X_train_mordred_2d_processed.csv", index_col=0
    )
elif descriptor_type == "mordred_3d":
    des = pd.read_csv(
        "outputs/descriptors/mordred_3d/X_train_mordred_3d_processed.csv", index_col=0
    )
elif descriptor_type == "fp":
    des = pd.read_csv("outputs/descriptors/fp/X_train_fp.csv", index_col=0)
else:
    raise ValueError(f"未知の descriptor_type: {descriptor_type}")

# 目的変数と記述子を結合
dataset_train = target.join(des, how="inner")
print(target.shape, des.shape, dataset_train.shape)

(13132, 2) (13132, 1382) (13132, 1384)


In [3]:
dataset_train.head()

,Emission max (nm),Quantum yield,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,LogEE_A,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
Tag,,,,,,,,,,,,,,,,,,,,,
72,538.0,0.200,2,0,37.051229,2.579777,5.159555,37.051229,1.277629,4.315072,...,10.630190,65.405606,641.695969,18.334171,1870,59,162.0,201.0,11.305556,6.194444
73,534.0,0.020,2,0,37.051229,2.579777,5.159555,37.051229,1.277629,4.315072,...,10.630190,65.405606,833.640512,23.818300,1870,59,162.0,201.0,11.305556,6.194444
74,566.0,0.018,2,0,41.407230,2.603784,5.207568,41.407230,1.254765,4.440806,...,10.833149,70.214037,969.484623,27.699561,2564,72,186.0,234.0,14.750000,7.000000
75,542.0,0.600,2,0,37.051229,2.579777,5.159555,37.051229,1.277629,4.315072,...,10.630190,65.405606,641.695969,18.334171,1870,59,162.0,201.0,11.305556,6.194444
76,545.0,0.080,2,0,37.051229,2.579777,5.159555,37.051229,1.277629,4.315072,...,10.630190,65.405606,833.640512,23.818300,1870,59,162.0,201.0,11.305556,6.194444


### テストデータ読み込み

In [4]:
# 入力：読み込みたい記述子のタイプを選択
descriptor_type = "mordred_3d"  # 'rdkit' or 'mordred_2d' or 'mordred_3d' or 'fp'

# dataset = pd.read_csv("../data/dataset_test.csv", index_col=0)

if descriptor_type == "rdkit":
    des = pd.read_csv("outputs/descriptors/rdkit/X_test_rdkit_processed.csv", index_col=0)
elif descriptor_type == "mordred_2d":
    des = pd.read_csv(
        "outputs/descriptors/mordred_2d/X_test_mordred_2d_processed.csv", index_col=0
    )
elif descriptor_type == "mordred_3d":
    des = pd.read_csv(
        "outputs/descriptors/mordred_3d/X_test_mordred_3d_processed.csv", index_col=0
    )
elif descriptor_type == "fp":
    des = pd.read_csv("outputs/descriptors/fp/X_test_fp.csv", index_col=0)
else:
    raise ValueError(f"未知の descriptor_type: {descriptor_type}")

# 目的変数と記述子を結合
X_test = des
print(des.shape, X_test.shape)

(7104, 1382) (7104, 1382)


In [5]:
X_test

,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,LogEE_A,VE1_A,VE2_A,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
Tag,,,,,,,,,,,,,,,,,,,,,
1,2,0,37.051229,2.579777,5.159555,37.051229,1.277629,4.315072,4.652435,0.160429,...,10.630190,65.405606,641.695969,18.334171,1870,59,162.0,201.0,11.305556,6.194444
2,2,0,37.051229,2.579777,5.159555,37.051229,1.277629,4.315072,4.652435,0.160429,...,10.630190,65.405606,833.640512,23.818300,1870,59,162.0,201.0,11.305556,6.194444
3,2,0,41.407230,2.603784,5.207568,41.407230,1.254765,4.440806,5.001998,0.151576,...,10.833149,70.214037,969.484623,27.699561,2564,72,186.0,234.0,14.750000,7.000000
4,2,0,37.051229,2.579777,5.159555,37.051229,1.277629,4.315072,4.652435,0.160429,...,10.630190,65.405606,641.695969,18.334171,1870,59,162.0,201.0,11.305556,6.194444
5,2,0,37.051229,2.579777,5.159555,37.051229,1.277629,4.315072,4.652435,0.160429,...,10.630190,65.405606,833.640512,23.818300,1870,59,162.0,201.0,11.305556,6.194444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20232,0,0,27.176227,2.668972,5.193255,27.176227,1.235283,4.053193,4.150208,0.188646,...,10.555526,74.138241,639.835229,17.773201,848,49,128.0,166.0,10.534722,4.666667
20233,0,0,27.176227,2.668972,5.193255,27.176227,1.235283,4.053193,4.150208,0.188646,...,10.555526,74.138241,639.835229,17.773201,848,49,128.0,166.0,10.534722,4.666667
20234,0,0,27.176227,2.668972,5.193255,27.176227,1.235283,4.053193,4.150208,0.188646,...,10.555526,74.138241,639.835229,17.773201,848,49,128.0,166.0,10.534722,4.666667


## 最終モデルの作成_emission予測モデル

### LightGBM ~学習データ全てを学習させる~

In [6]:
# 目的変数と説明変数
y = dataset_train["Emission max (nm)"]
X = dataset_train.drop("Emission max (nm)", axis=1)
X = X.drop("Quantum yield", axis=1)

print("X:", X.shape, "X_test:", X_test.shape)

X: (13132, 1382) X_test: (7104, 1382)


In [7]:
# オートスケーリング（学習データの統計量で fit）
X_scaler = StandardScaler()
autoscaled_X_train = pd.DataFrame(
    X_scaler.fit_transform(X),
    columns=X.columns,
    index=X.index,
)
autoscaled_X_test = pd.DataFrame(
    X_scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

y_scaler = StandardScaler()
autoscaled_y_train = y_scaler.fit_transform(y.values.reshape(-1, 1))

autoscaled_y_train = pd.DataFrame(
    autoscaled_y_train, index=y.index, columns=["y"]
)

In [8]:
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',  # LightGBMが内部で使う評価指標
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'random_state': 1234,
        'verbosity': -1,
        'device': 'cpu',
    }

    # 分割、CV
    kf = KFold(n_splits=5, shuffle=True, random_state=1234)

    # 評価指標
    rmse_scores = []

    for train_index, val_index in kf.split(X):
        # 訓練と検証に分類
        X_train, X_val = X.iloc[train_index, :], X.iloc[val_index, :]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        # 重要: スケーラーは各foldの train のみでfit（情報リークを防ぐ）
        X_scaler = StandardScaler()
        autoscaled_X_train = pd.DataFrame(
            X_scaler.fit_transform(X_train),
            columns=X_train.columns,
            index=X_train.index,
        )
        autoscaled_X_val = pd.DataFrame(
            X_scaler.transform(X_val),
            columns=X_val.columns,
            index=X_val.index,
        )

        y_scaler = StandardScaler()
        autoscaled_y_train = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

        # モデル定義・学習
        model_lgb = lgb.LGBMRegressor(**params)
        model_lgb.fit(autoscaled_X_train, autoscaled_y_train)

        # 予測（元スケールへ戻す）
        y_pred_scaled = model_lgb.predict(autoscaled_X_val)
        y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

        # 性能チェック
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        # 結果格納
        rmse_scores.append(rmse)

    return float(np.mean(rmse_scores))  # 最適化したい評価指標を選ぶ

In [9]:
# 最適化
study = optuna.create_study(direction='minimize', study_name='regression')
study.optimize(objective, n_trials=30)

# ハイパーパラメータ・スコアの確認
print("Best trial:")
trial = study.best_trial

print(f"  RMSE: {trial.value:.4f}")
print("  Params:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2026-04-26 22:12:14,523] A new study created in memory with name: regression
[I 2026-04-26 22:13:32,794] Trial 0 finished with value: 36.604301442074615 and parameters: {'n_estimators': 258, 'learning_rate': 0.08357488100159911, 'max_depth': 10, 'num_leaves': 64, 'subsample': 0.5724198185482007, 'colsample_bytree': 0.5802041774577025, 'reg_alpha': 0.000752097990720075, 'reg_lambda': 0.4260044470903096}. Best is trial 0 with value: 36.604301442074615.
[I 2026-04-26 22:13:56,035] Trial 1 finished with value: 38.72073950875454 and parameters: {'n_estimators': 90, 'learning_rate': 0.09322836199793036, 'max_depth': 9, 'num_leaves': 29, 'subsample': 0.7060259911645033, 'colsample_bytree': 0.5783333022461197, 'reg_alpha': 2.0978676706643835e-08, 'reg_lambda': 0.0025907079247550462}. Best is trial 0 with value: 36.604301442074615.
[I 2026-04-26 22:14:50,072] Trial 2 finished with value: 36.55233159531734 and parameters: {'n_estimators': 146, 'learning_rate': 0.09632806652288246, 'max_depth'

Best trial:
  RMSE: 36.3328
  Params:
    n_estimators: 177
    learning_rate: 0.0527852406960674
    max_depth: 8
    num_leaves: 99
    subsample: 0.9020255234131497
    colsample_bytree: 0.6732350819843801
    reg_alpha: 0.010549590779795488
    reg_lambda: 0.00040806187412644975


In [10]:
# optunaで最適化されたパラメータをセットし、再評価する
model_lgb_op = lgb.LGBMRegressor(**study.best_params, random_state=1234, verbosity=-1, device='cpu')

# 5分割交差検証
kf = KFold(n_splits=5, shuffle=True, random_state=1234)

# スコア保存用
rmse_scores = []
mae_scores = []
r2_scores = []

for train_index, val_index in kf.split(X):
    # 訓練と検証に分類
    X_train, X_val = X.iloc[train_index, :], X.iloc[val_index, :]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # 標準化、DataFrameに戻す
    X_scaler = StandardScaler()
    autoscaled_X_train = pd.DataFrame(X_scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    autoscaled_X_val = pd.DataFrame(X_scaler.transform(X_val), columns=X_val.columns, index=X_val.index)

    y_scaler = StandardScaler()
    autoscaled_y_train = pd.DataFrame(y_scaler.fit_transform(y_train.values.reshape(-1,1)), index=y_train.index, columns=['y'])
    autoscaled_y_val = pd.DataFrame(y_scaler.transform(y_val.values.reshape(-1,1)), index=y_val.index, columns=['y'])

    # 学習
    model_lgb_op.fit(autoscaled_X_train, autoscaled_y_train, eval_set=[(autoscaled_X_val, autoscaled_y_val)])
    y_pred_scaled = model_lgb_op.predict(autoscaled_X_val)
    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1,1))

    # 性能チェック
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)

    print(f'Fold RMSE : {rmse:.4f}')
    print(f'Fold MAE : {mae:.4f}')
    print(f'Fold R2 : {r2:.4f}')
    print()

print(f'平均RMSE : {np.mean(rmse_scores):.4f}')
print(f'平均MAE : {np.mean(mae_scores):.4f}')
print(f'平均R2 : {np.mean(r2_scores):.4f}')

Fold RMSE : 37.2348
Fold MAE : 24.6538
Fold R2 : 0.8436

Fold RMSE : 35.5605
Fold MAE : 23.8070
Fold R2 : 0.8512

Fold RMSE : 35.5659
Fold MAE : 24.0965
Fold R2 : 0.8584

Fold RMSE : 37.1335
Fold MAE : 25.0625
Fold R2 : 0.8473

Fold RMSE : 36.1695
Fold MAE : 24.7452
Fold R2 : 0.8591

平均RMSE : 36.3328
平均MAE : 24.4730
平均R2 : 0.8519


In [11]:
# 最終モデル構築（LGB）

# LGB本体（best_params を使用）
# 注意: lgb という変数名は lightgbm モジュール名と衝突するため使わない
lgb_emi = lgb.LGBMRegressor(**study.best_params, random_state=1234, verbosity=-1, device='cpu')

# X標準化 + LGB
x_pipe = Pipeline([
    ("x_scaler", StandardScaler()),
    ("lgb", lgb_emi),
])

# 重要: y標準化も含める（predict時は自動で元スケールに逆変換される）
model_lgb_final = TransformedTargetRegressor(
    regressor=x_pipe,
    transformer=StandardScaler()
)

# 学習
model_lgb_final.fit(X, y)

# 保存（descriptor_type を明示）
os.makedirs("models/lgb", exist_ok=True)

model_name = f"lgb_{descriptor_type}_emi"   # 例: lgb_mordred_3d_emi
save_path = f"models/lgb/artifact_{model_name}.joblib"

# 重要: 推論時に列順を再現できるよう、feature_columns も一緒に保存
artifact_lgb = {
    "model_name": model_name,
    "model_type": "LGB",
    "descriptor_type": descriptor_type,
    "model": model_lgb_final,
    "feature_columns": X.columns.tolist(),
}

joblib.dump(artifact_lgb, save_path)

print(f"saved: {save_path}")

saved: models/lgb/artifact_lgb_mordred_3d_emi.joblib


## 最終モデルの構築_quantum yield予測モデル

In [15]:
# 目的変数と説明変数（quantum yield）
y_qy = dataset_train["Quantum yield"]
X_qy = dataset_train.drop(columns=["Emission max (nm)", "Quantum yield"])

print("X_qy:", X_qy.shape, "X_test:", X_test.shape)

X_qy: (13132, 1382) X_test: (7104, 1382)


In [16]:
# 念のためモジュールを再束縛（過去セル実行での上書き対策）
import lightgbm as lgb

def objective_qy(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "random_state": 1234,
        "verbosity": -1,
        "device": "cpu",
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=1234)
    rmse_scores = []

    for train_index, val_index in kf.split(X_qy):
        X_train_qy = X_qy.iloc[train_index, :]
        X_val_qy = X_qy.iloc[val_index, :]
        y_train_qy = y_qy.iloc[train_index]
        y_val_qy = y_qy.iloc[val_index]

        X_scaler = StandardScaler()
        autoscaled_X_train_qy = pd.DataFrame(
            X_scaler.fit_transform(X_train_qy),
            columns=X_train_qy.columns,
            index=X_train_qy.index,
        )
        autoscaled_X_val_qy = pd.DataFrame(
            X_scaler.transform(X_val_qy),
            columns=X_val_qy.columns,
            index=X_val_qy.index,
        )

        y_scaler = StandardScaler()
        autoscaled_y_train_qy = y_scaler.fit_transform(
            y_train_qy.values.reshape(-1, 1)
        ).ravel()

        model_lgb_qy = lgb.LGBMRegressor(**params)
        model_lgb_qy.fit(autoscaled_X_train_qy, autoscaled_y_train_qy)

        y_pred_scaled_qy = model_lgb_qy.predict(autoscaled_X_val_qy)
        y_pred_qy = y_scaler.inverse_transform(
            y_pred_scaled_qy.reshape(-1, 1)
        ).ravel()

        rmse_scores.append(np.sqrt(mean_squared_error(y_val_qy, y_pred_qy)))

    return float(np.mean(rmse_scores))

In [17]:
# quantum yield モデルの最適化
study_qy = optuna.create_study(direction="minimize", study_name="regression_qy")
study_qy.optimize(objective_qy, n_trials=30)

print("Best trial (Quantum yield):")
trial_qy = study_qy.best_trial
print(f"  RMSE: {trial_qy.value:.4f}")
print("  Params:")
for key, value in trial_qy.params.items():
    print(f"    {key}: {value}")

[I 2026-04-27 18:59:15,816] A new study created in memory with name: regression_qy
[I 2026-04-27 18:59:32,851] Trial 0 finished with value: 0.27181802622986856 and parameters: {'n_estimators': 61, 'learning_rate': 0.018085960893222577, 'max_depth': 4, 'num_leaves': 17, 'subsample': 0.7287545705465781, 'colsample_bytree': 0.7411761771329992, 'reg_alpha': 1.2154246904992722e-06, 'reg_lambda': 0.0001723759696876485}. Best is trial 0 with value: 0.27181802622986856.
[I 2026-04-27 18:59:58,018] Trial 1 finished with value: 0.21432047491184775 and parameters: {'n_estimators': 248, 'learning_rate': 0.06154690527738564, 'max_depth': 4, 'num_leaves': 49, 'subsample': 0.5384834917857959, 'colsample_bytree': 0.6155687781512593, 'reg_alpha': 0.0013990370269339573, 'reg_lambda': 0.008307821405545597}. Best is trial 1 with value: 0.21432047491184775.
[I 2026-04-27 19:00:13,944] Trial 2 finished with value: 0.21124001047150426 and parameters: {'n_estimators': 62, 'learning_rate': 0.2768756271552285, 

Best trial (Quantum yield):
  RMSE: 0.2022
  Params:
    n_estimators: 298
    learning_rate: 0.052239251096013746
    max_depth: 8
    num_leaves: 37
    subsample: 0.824008344611545
    colsample_bytree: 0.6868670142071513
    reg_alpha: 0.04803483613974534
    reg_lambda: 3.439011901903068e-07


In [18]:
# 最適化パラメータで再評価（quantum yield）
model_lgb_op_qy = lgb.LGBMRegressor(
    **study_qy.best_params, random_state=1234, verbosity=-1, device="cpu"
)

kf_qy = KFold(n_splits=5, shuffle=True, random_state=1234)
rmse_scores_qy = []
mae_scores_qy = []
r2_scores_qy = []

for train_index, val_index in kf_qy.split(X_qy):
    X_train_qy = X_qy.iloc[train_index, :]
    X_val_qy = X_qy.iloc[val_index, :]
    y_train_qy = y_qy.iloc[train_index]
    y_val_qy = y_qy.iloc[val_index]

    X_scaler = StandardScaler()
    autoscaled_X_train_qy = pd.DataFrame(
        X_scaler.fit_transform(X_train_qy),
        columns=X_train_qy.columns,
        index=X_train_qy.index,
    )
    autoscaled_X_val_qy = pd.DataFrame(
        X_scaler.transform(X_val_qy),
        columns=X_val_qy.columns,
        index=X_val_qy.index,
    )

    y_scaler = StandardScaler()
    autoscaled_y_train_qy = y_scaler.fit_transform(
        y_train_qy.values.reshape(-1, 1)
    ).ravel()

    model_lgb_op_qy.fit(autoscaled_X_train_qy, autoscaled_y_train_qy)
    y_pred_scaled_qy = model_lgb_op_qy.predict(autoscaled_X_val_qy)
    y_pred_qy = y_scaler.inverse_transform(y_pred_scaled_qy.reshape(-1, 1)).ravel()

    rmse = np.sqrt(mean_squared_error(y_val_qy, y_pred_qy))
    mae = mean_absolute_error(y_val_qy, y_pred_qy)
    r2 = r2_score(y_val_qy, y_pred_qy)

    rmse_scores_qy.append(rmse)
    mae_scores_qy.append(mae)
    r2_scores_qy.append(r2)

    print(f"Fold RMSE : {rmse:.4f}")
    print(f"Fold MAE : {mae:.4f}")
    print(f"Fold R2 : {r2:.4f}\n")

print("=== Mean CV Scores (Quantum yield) ===")
print(f"RMSE: {np.mean(rmse_scores_qy):.4f}")
print(f"MAE : {np.mean(mae_scores_qy):.4f}")
print(f"R2  : {np.mean(r2_scores_qy):.4f}")

Fold RMSE : 0.2033
Fold MAE : 0.1471
Fold R2 : 0.5627

Fold RMSE : 0.2004
Fold MAE : 0.1464
Fold R2 : 0.5874

Fold RMSE : 0.2016
Fold MAE : 0.1469
Fold R2 : 0.5684

Fold RMSE : 0.2022
Fold MAE : 0.1473
Fold R2 : 0.5560

Fold RMSE : 0.2032
Fold MAE : 0.1482
Fold R2 : 0.5599

=== Mean CV Scores (Quantum yield) ===
RMSE: 0.2022
MAE : 0.1472
R2  : 0.5669


In [19]:
# 最終モデル構築と保存（quantum yield）
lgb_qy = lgb.LGBMRegressor(
    **study_qy.best_params, random_state=1234, verbosity=-1, device="cpu"
)

x_pipe_qy = Pipeline([
    ("x_scaler", StandardScaler()),
    ("lgb", lgb_qy),
])

model_lgb_final_qy = TransformedTargetRegressor(
    regressor=x_pipe_qy,
    transformer=StandardScaler(),
)

model_lgb_final_qy.fit(X_qy, y_qy)

os.makedirs("models/lgb", exist_ok=True)

model_name_qy = f"lgb_{descriptor_type}_qy"   # 例: lgb_mordred_3d_qy
save_path_qy = f"models/lgb/artifact_{model_name_qy}.joblib"

# 重要: emission モデルと衝突しないよう、_qy サフィックスで保存
artifact_lgb_qy = {
    "model_name": model_name_qy,
    "model_type": "LGB",
    "descriptor_type": descriptor_type,
    "target": "Quantum yield",
    "model": model_lgb_final_qy,
    "feature_columns": X_qy.columns.tolist(),
}

joblib.dump(artifact_lgb_qy, save_path_qy)
print(f"saved: {save_path_qy}")

saved: models/lgb/artifact_lgb_mordred_3d_qy.joblib
